# 自研 Decoder：Dense / MoE 的 BF16 训练与恢复

这里使用新的 `lm.Config` / `lm.Decoder`，与字符教学模型区分开。数据是固定整数序列，不依赖外部 tokenizer、语料或下载。

两种结构各真实训练 20 步，展示 BF16 激活、FP32 参数/Adam、梯度裁剪、分块输出头，以及完整 checkpoint 后的同一下一步恢复。小数据上的 loss 下降只说明流程可学习，不代表语言能力。

In [ ]:
from itertools import cycle
from tempfile import TemporaryDirectory

from qutai import lm, nn, visualization as vis

device = "cuda"
dtype = "float32"
precision = "bfloat16" if device == "cuda" else None
steps = 20
sequence_length = 33
inputs = [[(position + offset) % 32 for position in range(sequence_length)] for offset in (0, 7)]
# labels 保持未shift；Decoder.loss内部只做一次next-token shift。
labels = [row[:] for row in inputs]


## 1. 同一个接口配置 Dense 与稀疏 MoE

两者都使用 RMSNorm、RoPE、GQA、SwiGLU 和共享 embedding/head。MoE 每个 token 只选两个专家；总参数与活跃参数分别显示。

`D128 / H2 / KV1` 对应 head_dim=64。CUDA 下用 `precision="bfloat16"`，中间激活实际占两个字节，模型参数、参数梯度、Adam 状态及稳定归约保持 FP32。

In [ ]:
def make_config(kind):
    return lm.Config(vocab_size=64, n_layers=2, d_model=128, n_heads=2, n_kv_heads=1,
                     d_ff=256, max_seq_len=64, ffn_type=kind, n_experts=4, top_k=2,
                     d_expert=128, activation_checkpointing=True)


def make_optimizer(model):
    parameters = model.parameters()
    return nn.AdamW([
        {"params": [p for p in parameters if p.ndim >= 2], "weight_decay": 0.1},
        {"params": [p for p in parameters if p.ndim < 2], "weight_decay": 0},
    ], learning_rate=0.003, betas=(0.9, 0.95))


vis.Table([[kind, *make_config(kind).parameter_counts().values()] for kind in ("dense", "moe")],
          columns=["结构", "总参数", "活跃参数"], title="本页的小模型规格")


## 2. 各训练20步

`lm.train` 是普通生成器，每次完成清梯度、前向、反向、裁剪和一次 AdamW 更新后产生 metrics。`chunk_size=16` 限制输出头暂存的 logits 行数，避免一次保存整个词表矩阵。

语料中两个序列循环使用是本页明确的 toy 任务；正式训练可接 `TokenShards` / `StatefulBatcher`，见同目录 `train.py`。

In [ ]:
experiments = {}
for kind in ("dense", "moe"):
    config = make_config(kind)
    model = lm.Decoder(config, device=device, dtype=dtype, seed=42)
    optimizer = make_optimizer(model)
    history = list(lm.train(model, optimizer, cycle([(inputs, labels)]), steps=steps,
                           precision=precision, max_grad_norm=1.0, chunk_size=16))
    experiments[kind] = (model, optimizer, history)
    print(kind, "loss", history[0]["loss"], "→", history[-1]["loss"])
    print("参数/Adam精度：", sorted({p.dtype for p in model.parameters()}),
          sorted({state.dtype for state in optimizer.averages.values()}))

loss_plot = vis.Plot(title="固定小序列的20步训练", xlabel="完整更新次数", ylabel="loss")
for kind, (_, _, history) in experiments.items():
    loss_plot.line([r["global_step"] for r in history], [r["loss"] for r in history], label=kind)
loss_plot


## 3. 保存、独立恢复，再走同一个下一步

最后的 MoE 用标准库临时目录保存，结束后自动清理，不触碰任何已有 checkpoint。恢复会原地写回参数与 Adam 状态。

这里的下一批数据固定，因此只需在状态中记录本页步数；正式语料要额外传入 `data=batcher` 和 tokenizer，才能一起恢复游标与指纹。

In [ ]:
model, optimizer, history = experiments["moe"]
restored = lm.Decoder(make_config("moe"), device=device, dtype=dtype, seed=99)
restored_optimizer = make_optimizer(restored)
with TemporaryDirectory(prefix="qutai-decoder-example-") as folder:
    lm.save_checkpoint(folder, model, optimizer, global_step=steps,
                       tokens=steps * len(inputs) * (sequence_length - 1),
                       learning_rate=optimizer.learning_rate, state={"toy_task": "cycle32"})
    state = lm.load_checkpoint(folder, restored, restored_optimizer)
    continuous = lm.train_step(model, optimizer, cycle([(inputs, labels)]),
                               precision=precision, max_grad_norm=1.0, chunk_size=16)
    resumed = lm.train_step(restored, restored_optimizer, cycle([(inputs, labels)]),
                            precision=precision, max_grad_norm=1.0, chunk_size=16)

print("保存步数：", state["global_step"])
print("同一下一步loss：", continuous["loss"], resumed["loss"])
assert continuous["loss"] == resumed["loss"]
assert all(a.storage.tolist() == b.storage.tolist()
           for a, b in zip(model.parameters(), restored.parameters()))
for a, b in zip(model.parameters(), restored.parameters()):
    assert optimizer.steps.get(a) == restored_optimizer.steps.get(b)
    assert (a in optimizer.averages) == (b in restored_optimizer.averages)
    if a in optimizer.averages:
        assert optimizer.averages[a].tolist() == restored_optimizer.averages[b].tolist()
        assert optimizer.square_averages[a].tolist() == restored_optimizer.square_averages[b].tolist()
print("参数、Adam动量、逐参数计步全部一致")


## 4. 接入真实语料

`configs/lm_debug.json`、`dense300m.json`、`moe1b.json` 给出三套结构。真实训练使用 `train.py`，需要显式指定 tokenizer、语料和新的输出目录，例如在项目根运行：

```text
python examples/lm/train.py --config examples/lm/configs/lm_debug.json --tokenizer path/to/tokenizer.json --data path/to/pretrain --output runs/debug --device cuda --precision bfloat16 --batch-size 2 --sequence-length 128 --max-steps 1000 --checkpoint-every 100
```

原 `language_model.ipynb` 展示字符编码和生成/KV cache；本页的 Decoder 侧重训练，不将旧模型的生成接口混算到新 Decoder 上。